In [0]:
from pyspark.sql import functions as F

# ---------------- CONFIG ----------------
BRONZE_JIRA = "workspace.slainte_bronze.jira_issues_bronze"
GOLD_DB     = "workspace.slainte_gold"
DIM_STATUS  = f"{GOLD_DB}.dim_status"

# ---------------- SETUP ----------------
spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

# ---------------- BUILD DIM ----------------
df_dim_status = (
    spark.table(BRONZE_JIRA)
    .select(
        F.col("user_id").alias("source_user_id"),
        F.col("project_key").alias("project"),
        F.upper(F.trim(F.col("status"))).alias("status_code")
    )
    .filter(
        F.col("user_id").isNotNull() &
        F.col("project_key").isNotNull() &
        F.col("status").isNotNull()
    )
    .distinct()
    .withColumn("status_label", F.initcap(F.regexp_replace(F.col("status_code"), "_", " ")))
    .withColumn("status_id", F.monotonically_increasing_id() + 1)
    .select("status_id", "source_user_id", "project", "status_code", "status_label")
)

# ---------------- WRITE ----------------
df_dim_status.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_STATUS)

print("✅ dim_status created successfully")
